# 02 — ST Population Sweep

Generates the SPEC §7.3 deliverables for the ST (`WindowedSpatioTemporalGATNet`) architecture across the full matrix:

- **3 regimes:** `kfold-5`, `kfold-10`, `loso`
- **2 mt:** 2, 4
- **3 chromophores:** `hbo`, `hbr`, `hbt`  *(rev. 6 — was HbO-only in rev. 5)*
- **2 paths:**
  - `native` — built-in `model.explain()` with reductions (heads → layers → α_k weighting → symmetric pair matrix → row-sum). SPEC §6.1 primary path. Essentially free at inference.
  - `supplementary` — GNNExplainer with `node_mask_type='object'` (mask is `[23]`, not `[23, 326]`). SPEC §6.4 cross-check.

= **36 PopulationResults** total per full sweep (3 regimes × 2 mt × 3 chromophores × 2 paths). Each lands at `research/xai/st/{hb}/{regime}/mt{N}/{path}/` with the §7.3 deliverables, including ST-only `temporal_attention.csv`.

> **Migration note (rev. 6, 2026-05-10).** ST checkpoints moved from `research/experiments/20260501/spatial_temporal_graph/` (non-uniform config across regimes) to `research/experiments/20260509/` (uniform config + 3 chromophores). LOSO accuracy improved by **+5.5 pp** mean (max +9.7 pp on HbR mt2). 5-fold/10-fold are within run-to-run noise. See `research/experiments/20260509/CONFIG_VS_BASELINE_REPORT.md` for the full comparison.
>
> The new kfold subtree has an extra date-named folder (`st-kfold/{5,10}-fold/20260509/<exp_dir>`); `XAIRunConfig.experiment_subdir` carries this override per cell. LOSO under 20260509 still matches the canonical `loso/` layout, so no override is needed.

> **Runtime note.** Native attention is essentially free (~30 s/regime/chromophore). Supplementary GNNExplainer ≈ 1 s/trial at `epochs=200` on GPU. `loso × supplementary × 3 chromophores` ≈ ~25–35 min per regime/mt cell. Pass `chromophores=("hbo",)` to skip HbR/HbT, or `run_supplementary=False` to skip the GNN object-mask path.

> **PyG 2.7 quirk handled in the supplementary path:** cuDNN's GRU backward refuses eval-mode gradients, so the explainer call is wrapped in `torch.backends.cudnn.flags(enabled=False)`. See `src/xai/st_explainer.py:explain_supplementary_checkpoint`.

> **Downstream notebooks need re-run after this sweep.** Both `03_cross_arch_comparison.ipynb` and `04_atlas_registration.ipynb` consume CSVs from `research/xai/st/...`; the new `{hb}/` directory layer means their globs may also need a small update before re-execution.

Reference: [`docs/SPEC_xai_graph.md`](../../../docs/SPEC_xai_graph.md) (rev. 6).

In [1]:
%matplotlib inline
import os, sys
from pathlib import Path

import numpy as np
import torch
from scipy import stats

PROJECT_ROOT = Path(os.getcwd()).resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xai import (
    XAIRunConfig,
    run_st, run_st_supplementary,
    plot_montage_channel_importance, plot_pair_matrix, plot_temporal_attention,
    write_run_json,
    PopulationResult,
    CHANNEL_NAMES, N_CH,
)

# rev. 6 layout — see header migration note.
ST_EXPERIMENT_ROOT = PROJECT_ROOT / "research/experiments/20260509"

# kfold under 20260509 has an extra date-named subdir; LOSO does not.
ST_KFOLD_SUBDIRS = {
    "kfold-5":  "st-kfold/5-fold/20260509",
    "kfold-10": "st-kfold/10-fold/20260509",
}

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"PROJECT_ROOT       = {PROJECT_ROOT}")
print(f"ST_EXPERIMENT_ROOT exists: {ST_EXPERIMENT_ROOT.is_dir()}")
print(f"  kfold-5  subdir: {(ST_EXPERIMENT_ROOT / ST_KFOLD_SUBDIRS['kfold-5']).is_dir()}")
print(f"  kfold-10 subdir: {(ST_EXPERIMENT_ROOT / ST_KFOLD_SUBDIRS['kfold-10']).is_dir()}")
print(f"  loso     subdir: {(ST_EXPERIMENT_ROOT / 'loso').is_dir()}")
print(f"DEVICE             = {DEVICE}")

PROJECT_ROOT       = /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method
ST_EXPERIMENT_ROOT exists: True
  kfold-5  subdir: True
  kfold-10 subdir: True
  loso     subdir: True
DEVICE             = cuda:0


## Helper

`run_st_cell(regime, mt, chromophores=("hbo","hbr","hbt"), run_supplementary, gnn_epochs)`
runs the native attention path (always) and the supplementary GNN object-mask path
(when `run_supplementary=True`) for one `(regime, mt)` matrix cell across the
specified chromophores. Persists everything to disk and returns a nested dict
`{chromophore: {path: PopulationResult}}`.

To re-run only HbO (the rev. 5 scope), pass `chromophores=("hbo",)`.

In [2]:
def run_st_cell(
    regime: str,
    mt: int,
    chromophores: tuple = ("hbo", "hbr", "hbt"),
    run_supplementary: bool = True,
    gnn_epochs: int = 200,
    gnn_lr: float = 0.01,
    head_reduce: str = "mean",
    layer_reduce: str = "mean",
) -> dict:
    """Run native + (optional) supplementary across chromophores for one (regime, mt) cell.

    Returns nested dict {chromophore: {path: PopulationResult}}.
    """
    out: dict = {}
    subdir = ST_KFOLD_SUBDIRS.get(regime)  # None for LOSO

    for hb in chromophores:
        out[hb] = {}

        # --- native attention (always) ---------------------------------------
        native_dir = PROJECT_ROOT / f"research/xai/st/{hb}/{regime}/mt{mt}/native"
        native_dir.mkdir(parents=True, exist_ok=True)
        cfg_native = XAIRunConfig(
            arch="st", hb=hb, regime=regime, mt=mt,
            experiment_root=str(ST_EXPERIMENT_ROOT),
            output_dir=str(native_dir),
            device=DEVICE,
            head_reduce=head_reduce,
            layer_reduce=layer_reduce,
            seed=42,
            experiment_subdir=subdir,
        )
        print(f"[{regime} mt={mt} {hb}] {'native':>13s} → {native_dir.relative_to(PROJECT_ROOT)} ...")
        res_native = run_st(cfg_native)
        res_native.to_csv(native_dir)
        write_run_json(cfg_native, res_native, native_dir)
        plot_montage_channel_importance(res_native, native_dir)
        plot_pair_matrix(res_native, native_dir)
        plot_temporal_attention(res_native, native_dir)
        out[hb]["native"] = res_native
        top5 = [CHANNEL_NAMES[i] for i in (-res_native.channel_importance_mean).argsort()[:5]]
        print(f"               done — n_trials={res_native.n_trials}/{res_native.n_trials_total} "
              f"({res_native.included_pct:.1f}%), top-5: {top5}")

        if run_supplementary:
            sup_dir = PROJECT_ROOT / f"research/xai/st/{hb}/{regime}/mt{mt}/supplementary"
            sup_dir.mkdir(parents=True, exist_ok=True)
            cfg_sup = XAIRunConfig(
                arch="st", hb=hb, regime=regime, mt=mt,
                experiment_root=str(ST_EXPERIMENT_ROOT),
                output_dir=str(sup_dir),
                device=DEVICE,
                gnn_explainer_epochs=gnn_epochs,
                gnn_explainer_lr=gnn_lr,
                run_supplementary_gnnexplainer=True,
                seed=42,
                experiment_subdir=subdir,
            )
            print(f"[{regime} mt={mt} {hb}] {'supplementary':>13s} → {sup_dir.relative_to(PROJECT_ROOT)} ...")
            res_sup = run_st_supplementary(cfg_sup)
            res_sup.to_csv(sup_dir)
            write_run_json(cfg_sup, res_sup, sup_dir)
            plot_montage_channel_importance(res_sup, sup_dir)
            plot_pair_matrix(res_sup, sup_dir)
            out[hb]["supplementary"] = res_sup
            top5 = [CHANNEL_NAMES[i] for i in (-res_sup.channel_importance_mean).argsort()[:5]]
            print(f"               done — top-5: {top5}")

    return out


def report_c5_agreement(results: dict, label: str) -> None:
    """SPEC §11 C5: per-chromophore Spearman ρ between native and supplementary top-10."""
    for hb, paths in results.items():
        if "supplementary" not in paths:
            print(f"[{label} {hb:>3s}] C5 skipped — supplementary path not run")
            continue
        a = paths["native"].channel_importance_mean
        b = paths["supplementary"].channel_importance_mean
        ra = (-a).argsort().argsort()
        rb = (-b).argsort().argsort()
        rho, _ = stats.spearmanr(ra, rb)
        flag = "✓" if rho >= 0.4 else "✗"
        print(f"[{label} {hb:>3s}] C5 — ρ(native top-10, supplementary top-10) = {rho:+.3f}   {flag} (≥ 0.4)")

## kfold-5 × mt=2

Native ≈ 30 s × 3 chromophores. Supplementary ≈ 3–5 min × 3 chromophores. ~15–20 min total.

In [3]:
st_kfold5_mt2 = run_st_cell(regime="kfold-5", mt=2)
report_c5_agreement(st_kfold5_mt2, "kfold-5 × mt=2")

[kfold-5 mt=2 hbo]        native → research/xai/st/hbo/kfold-5/mt2/native ...
               done — n_trials=93/124 (75.0%), top-5: ['S3_D3', 'S4_D4', 'S7_D4', 'S5_D5', 'S8_D5']
[kfold-5 mt=2 hbo] supplementary → research/xai/st/hbo/kfold-5/mt2/supplementary ...
               done — top-5: ['S1_D3', 'S4_D7', 'S3_D6', 'S8_D5', 'S7_D6']
[kfold-5 mt=2 hbr]        native → research/xai/st/hbr/kfold-5/mt2/native ...
               done — n_trials=92/124 (74.2%), top-5: ['S3_D3', 'S4_D4', 'S5_D5', 'S7_D7', 'S8_D5']
[kfold-5 mt=2 hbr] supplementary → research/xai/st/hbr/kfold-5/mt2/supplementary ...
               done — top-5: ['S3_D6', 'S4_D7', 'S1_D3', 'S8_D5', 'S3_D1']
[kfold-5 mt=2 hbt]        native → research/xai/st/hbt/kfold-5/mt2/native ...
               done — n_trials=96/124 (77.4%), top-5: ['S4_D4', 'S3_D3', 'S5_D5', 'S8_D5', 'S7_D6']
[kfold-5 mt=2 hbt] supplementary → research/xai/st/hbt/kfold-5/mt2/supplementary ...
               done — top-5: ['S3_D6', 'S1_D3', 'S4_D7', 'S6_

## kfold-5 × mt=4

Native ≈ 1 min × 3 chromophores. Supplementary ≈ 6–10 min × 3 chromophores. ~25–35 min total.

In [4]:
st_kfold5_mt4 = run_st_cell(regime="kfold-5", mt=4)
report_c5_agreement(st_kfold5_mt4, "kfold-5 × mt=4")

[kfold-5 mt=4 hbo]        native → research/xai/st/hbo/kfold-5/mt4/native ...
               done — n_trials=182/248 (73.4%), top-5: ['S2_D2', 'S7_D6', 'S5_D5', 'S3_D1', 'S8_D5']
[kfold-5 mt=4 hbo] supplementary → research/xai/st/hbo/kfold-5/mt4/supplementary ...
               done — top-5: ['S3_D6', 'S1_D3', 'S4_D7', 'S8_D5', 'S6_D3']
[kfold-5 mt=4 hbr]        native → research/xai/st/hbr/kfold-5/mt4/native ...
               done — n_trials=172/248 (69.4%), top-5: ['S7_D7', 'S8_D5', 'S5_D5', 'S8_D7', 'S3_D3']
[kfold-5 mt=4 hbr] supplementary → research/xai/st/hbr/kfold-5/mt4/supplementary ...
               done — top-5: ['S8_D5', 'S1_D3', 'S3_D6', 'S6_D3', 'S4_D7']
[kfold-5 mt=4 hbt]        native → research/xai/st/hbt/kfold-5/mt4/native ...
               done — n_trials=177/248 (71.4%), top-5: ['S2_D2', 'S1_D1', 'S7_D6', 'S8_D5', 'S3_D1']
[kfold-5 mt=4 hbt] supplementary → research/xai/st/hbt/kfold-5/mt4/supplementary ...
               done — top-5: ['S1_D3', 'S4_D7', 'S8_D5', '

## kfold-10 × mt=2

Similar to kfold-5 mt2 in total runtime (same number of trials, more folds). ~15–20 min total across 3 chromophores.

In [5]:
st_kfold10_mt2 = run_st_cell(regime="kfold-10", mt=2)
report_c5_agreement(st_kfold10_mt2, "kfold-10 × mt=2")

[kfold-10 mt=2 hbo]        native → research/xai/st/hbo/kfold-10/mt2/native ...
               done — n_trials=95/124 (76.6%), top-5: ['S3_D3', 'S7_D7', 'S8_D5', 'S7_D4', 'S4_D4']
[kfold-10 mt=2 hbo] supplementary → research/xai/st/hbo/kfold-10/mt2/supplementary ...
               done — top-5: ['S4_D7', 'S1_D3', 'S3_D6', 'S8_D5', 'S3_D1']
[kfold-10 mt=2 hbr]        native → research/xai/st/hbr/kfold-10/mt2/native ...
               done — n_trials=95/124 (76.6%), top-5: ['S8_D5', 'S4_D4', 'S7_D4', 'S5_D5', 'S3_D3']
[kfold-10 mt=2 hbr] supplementary → research/xai/st/hbr/kfold-10/mt2/supplementary ...
               done — top-5: ['S1_D3', 'S3_D1', 'S3_D6', 'S8_D5', 'S6_D3']
[kfold-10 mt=2 hbt]        native → research/xai/st/hbt/kfold-10/mt2/native ...
               done — n_trials=96/124 (77.4%), top-5: ['S4_D4', 'S3_D3', 'S8_D5', 'S7_D6', 'S5_D5']
[kfold-10 mt=2 hbt] supplementary → research/xai/st/hbt/kfold-10/mt2/supplementary ...
               done — top-5: ['S1_D3', 'S3_D6', '

## kfold-10 × mt=4

Native ≈ 1 min × 3 chromophores. Supplementary ≈ 10–15 min × 3 chromophores. ~30–50 min total.

In [6]:
st_kfold10_mt4 = run_st_cell(regime="kfold-10", mt=4)
report_c5_agreement(st_kfold10_mt4, "kfold-10 × mt=4")

[kfold-10 mt=4 hbo]        native → research/xai/st/hbo/kfold-10/mt4/native ...
               done — n_trials=186/248 (75.0%), top-5: ['S8_D7', 'S8_D5', 'S5_D5', 'S7_D7', 'S3_D3']
[kfold-10 mt=4 hbo] supplementary → research/xai/st/hbo/kfold-10/mt4/supplementary ...
               done — top-5: ['S3_D6', 'S4_D7', 'S1_D3', 'S8_D5', 'S6_D3']
[kfold-10 mt=4 hbr]        native → research/xai/st/hbr/kfold-10/mt4/native ...
               done — n_trials=183/248 (73.8%), top-5: ['S5_D5', 'S7_D7', 'S8_D5', 'S3_D1', 'S5_D8']
[kfold-10 mt=4 hbr] supplementary → research/xai/st/hbr/kfold-10/mt4/supplementary ...
               done — top-5: ['S3_D6', 'S1_D3', 'S4_D7', 'S8_D5', 'S6_D3']
[kfold-10 mt=4 hbt]        native → research/xai/st/hbt/kfold-10/mt4/native ...
               done — n_trials=184/248 (74.2%), top-5: ['S8_D7', 'S5_D5', 'S3_D1', 'S3_D3', 'S8_D5']
[kfold-10 mt=4 hbt] supplementary → research/xai/st/hbt/kfold-10/mt4/supplementary ...
               done — top-5: ['S3_D6', 'S1_D3'

## LOSO × mt=2

⚠ **Long-running supplementary across 3 chromophores.** 62 subjects × ~2 val trials × 200 epochs ≈ 5–10 min per chromophore. Native is fast (~1 min/chromophore). Total ≈ 15–35 min.

In [7]:
st_loso_mt2 = run_st_cell(regime="loso", mt=2)
report_c5_agreement(st_loso_mt2, "loso × mt=2")

[loso mt=2 hbo]        native → research/xai/st/hbo/loso/mt2/native ...
               done — n_trials=98/124 (79.0%), top-5: ['S5_D5', 'S7_D4', 'S4_D4', 'S8_D5', 'S1_D1']
[loso mt=2 hbo] supplementary → research/xai/st/hbo/loso/mt2/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S8_D5', 'S2_D1', 'S7_D6']
[loso mt=2 hbr]        native → research/xai/st/hbr/loso/mt2/native ...
               done — n_trials=102/124 (82.3%), top-5: ['S4_D4', 'S3_D3', 'S5_D5', 'S7_D4', 'S8_D5']
[loso mt=2 hbr] supplementary → research/xai/st/hbr/loso/mt2/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S2_D1', 'S8_D5', 'S7_D4']
[loso mt=2 hbt]        native → research/xai/st/hbt/loso/mt2/native ...
               done — n_trials=97/124 (78.2%), top-5: ['S4_D4', 'S5_D5', 'S3_D3', 'S7_D4', 'S8_D5']
[loso mt=2 hbt] supplementary → research/xai/st/hbt/loso/mt2/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S8_D5', 'S2_D1', 'S7_D6']
[loso × mt=2 hbo] C5 

## LOSO × mt=4

⚠ **Longest cell.** 62 subjects × ~4 val trials × 3 chromophores. Supplementary ≈ 15–25 min per chromophore. Total ≈ 50–80 min.

In [8]:
st_loso_mt4 = run_st_cell(regime="loso", mt=4)
report_c5_agreement(st_loso_mt4, "loso × mt=4")

[loso mt=4 hbo]        native → research/xai/st/hbo/loso/mt4/native ...
               done — n_trials=191/248 (77.0%), top-5: ['S8_D5', 'S2_D2', 'S3_D3', 'S5_D5', 'S2_D5']
[loso mt=4 hbo] supplementary → research/xai/st/hbo/loso/mt4/supplementary ...
               done — top-5: ['S1_D3', 'S3_D6', 'S2_D1', 'S3_D1', 'S7_D6']
[loso mt=4 hbr]        native → research/xai/st/hbr/loso/mt4/native ...
               done — n_trials=202/248 (81.5%), top-5: ['S7_D6', 'S5_D5', 'S3_D3', 'S3_D1', 'S3_D6']
[loso mt=4 hbr] supplementary → research/xai/st/hbr/loso/mt4/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S8_D5', 'S3_D6', 'S4_D7']
[loso mt=4 hbt]        native → research/xai/st/hbt/loso/mt4/native ...
               done — n_trials=187/248 (75.4%), top-5: ['S5_D5', 'S7_D6', 'S2_D2', 'S8_D5', 'S3_D3']
[loso mt=4 hbt] supplementary → research/xai/st/hbt/loso/mt4/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S8_D5', 'S4_D7', 'S2_D1']
[loso × mt=4 hbo] C

## Summary

Top-5 channels per cell, mt=2 ↔ mt=4 stability per path (SPEC §11 C3 ≥ 0.5), and a quick look at the temporal attention concentration (peak window per regime/mt).

In [ ]:
ALL_RUNS = {
    "kfold-5/mt2":  st_kfold5_mt2,
    "kfold-5/mt4":  st_kfold5_mt4,
    "kfold-10/mt2": st_kfold10_mt2,
    "kfold-10/mt4": st_kfold10_mt4,
    "loso/mt2":     st_loso_mt2,
    "loso/mt4":     st_loso_mt4,
}

# Each ALL_RUNS[cell] is a {chromophore: {path: PopulationResult}} dict.
ALL_HBS = sorted({hb for by_hb in ALL_RUNS.values() for hb in by_hb.keys()})

print("=" * 84)
print(f"Top-5 channels per (regime, mt, hb, path)   |   chromophores: {ALL_HBS}")
print("=" * 84)
for cell, by_hb in ALL_RUNS.items():
    for hb in ALL_HBS:
        if hb not in by_hb:
            continue
        for path, r in by_hb[hb].items():
            top5 = [CHANNEL_NAMES[i] for i in (-r.channel_importance_mean).argsort()[:5]]
            print(f"  {cell:>12s} | {hb:>3s} | {path:>13s} : {top5}")

print()
print("=" * 84)
print("SPEC §11 C3 — mt=2 vs mt=4 stability per regime/hb/path (ρ ≥ 0.5)")
print("=" * 84)
for regime in ("kfold-5", "kfold-10", "loso"):
    mt2 = ALL_RUNS[f"{regime}/mt2"]
    mt4 = ALL_RUNS[f"{regime}/mt4"]
    for hb in ALL_HBS:
        if hb not in mt2 or hb not in mt4:
            continue
        for path in ("native", "supplementary"):
            if path not in mt2[hb] or path not in mt4[hb]:
                continue
            r2 = (-mt2[hb][path].channel_importance_mean).argsort().argsort()
            r4 = (-mt4[hb][path].channel_importance_mean).argsort().argsort()
            rho, _ = stats.spearmanr(r2, r4)
            flag = "✓" if rho >= 0.5 else "✗"
            print(f"  {regime:>9s} {hb:>3s} {path:>13s}: ρ(mt2, mt4) = {rho:+.3f}   {flag}")

print()
print("=" * 84)
print("Temporal attention — peak window per (regime, mt, hb)")
print("=" * 84)
for cell, by_hb in ALL_RUNS.items():
    for hb in ALL_HBS:
        if hb not in by_hb or "native" not in by_hb[hb]:
            continue
        r = by_hb[hb]["native"]
        peak_idx = int(np.argmax(r.temporal_attention_mean))
        if r.window_times is not None:
            t_start, t_end = r.window_times[peak_idx]
            print(f"  {cell:>12s} {hb:>3s} : peak window k={peak_idx} → [{t_start:.1f}–{t_end:.1f}] s, "
                  f"α={r.temporal_attention_mean[peak_idx]:.3f}ßß")
        else:
            print(f"  {cell:>12s} {hb:>3s} : peak window k={peak_idx}, α={r.temporal_attention_mean[peak_idx]:.3f}")

Top-5 channels per (regime, mt, hb, path)   |   chromophores: ['hbo', 'hbr', 'hbt']
   kfold-5/mt2 | hbo |        native : ['S3_D3', 'S4_D4', 'S7_D4', 'S5_D5', 'S8_D5']
   kfold-5/mt2 | hbo | supplementary : ['S1_D3', 'S4_D7', 'S3_D6', 'S8_D5', 'S7_D6']
   kfold-5/mt2 | hbr |        native : ['S3_D3', 'S4_D4', 'S5_D5', 'S7_D7', 'S8_D5']
   kfold-5/mt2 | hbr | supplementary : ['S3_D6', 'S4_D7', 'S1_D3', 'S8_D5', 'S3_D1']
   kfold-5/mt2 | hbt |        native : ['S4_D4', 'S3_D3', 'S5_D5', 'S8_D5', 'S7_D6']
   kfold-5/mt2 | hbt | supplementary : ['S3_D6', 'S1_D3', 'S4_D7', 'S6_D3', 'S7_D6']
   kfold-5/mt4 | hbo |        native : ['S2_D2', 'S7_D6', 'S5_D5', 'S3_D1', 'S8_D5']
   kfold-5/mt4 | hbo | supplementary : ['S3_D6', 'S1_D3', 'S4_D7', 'S8_D5', 'S6_D3']
   kfold-5/mt4 | hbr |        native : ['S7_D7', 'S8_D5', 'S5_D5', 'S8_D7', 'S3_D3']
   kfold-5/mt4 | hbr | supplementary : ['S8_D5', 'S1_D3', 'S3_D6', 'S6_D3', 'S4_D7']
   kfold-5/mt4 | hbt |        native : ['S2_D2', 'S1_D1', 'S7_D6',

## Done

Outputs are persisted to `research/xai/st/{hb}/{regime}/mt{N}/{native|supplementary}/` and ready for `03_cross_arch_comparison.ipynb` and `04_atlas_registration.ipynb`. The C5 (native vs supplementary cross-check) and C3 (mt stability) acceptance checks are reported above; C2 (across-fold), C4 (3-way for SG), and C6 (biological-prior overlap) live in `03_cross_arch_comparison.ipynb`.

> **Downstream re-run reminder.** The `{hb}/` layer in the output path is new in rev. 6. `03_cross_arch_comparison.ipynb` and `04_atlas_registration.ipynb` (which autodiscovers cells from on-disk CSVs) both consume from `research/xai/st/...` and may need their globs updated to walk the new chromophore layer before re-execution.